# Your prepared video — Kaggle GPU render
In notebook Settings select **GPU T4 ×2** if available and turn **Internet ON**, then press **Run all**. Accelerator availability depends on your Kaggle account and quota.
The prepared project and soundtrack download automatically from the public repo and are hash-verified. No file uploads, API tokens or datasets are required. Blender and character downloads also run automatically.
The finished MP4 previews below and appears as a download link and in `/kaggle/working/video_engine/outputs`. If rendering fails, download the error-report ZIP and send it back.
Current project: **The Last Cookie**, an original 7.4-second test, 1080x1920 / 30 fps. The live Kaggle CUDA render has not yet been tested.
Character: Snow Rig © Blender Foundation | studio.blender.org — CC BY 4.0.

In [ ]:
# Render the prepared video on a Kaggle GPU
from pathlib import Path
import base64, collections, hashlib, io, json, subprocess, sys, time, zipfile
from IPython.display import Video, display
from IPython.display import FileLink
import os, shutil, urllib.request
os.chdir('/kaggle/working')
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Enable a GPU in Kaggle Settings > Accelerator, then Run all again.')
try:
    with urllib.request.urlopen('https://download.blender.org/release/Blender5.2/blender-5.2.3.sha256', timeout=30) as response:
        response.read(128)
except Exception as error:
    raise RuntimeError('Enable Internet in Kaggle notebook Settings, then Run all again.') from error
def download_link(path):
    relative = Path(path).resolve().relative_to(Path('/kaggle/working').resolve())
    display(FileLink(relative.as_posix()))
    print('Also available in the Kaggle Output / working-files panel:', relative.as_posix())
EXPECTED_SHA256 = 'e8a24682d9d4480567aa1fb76d04d5cabd7e355077d3ee6c0cbb8ed7c2ce18d5'
BUNDLE_URL = 'https://raw.githubusercontent.com/itskrishnamalhotra-stack/colab-video-engine/main/render-bundle.zip'
with urllib.request.urlopen(BUNDLE_URL, timeout=120) as response:
    data = response.read()
if hashlib.sha256(data).hexdigest() != EXPECTED_SHA256:
    raise ValueError("The embedded package is damaged. Reopen the latest notebook.")
destination = Path('/kaggle/working').resolve()
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    if any(not (destination/n).resolve().is_relative_to(destination) for n in archive.namelist()):
        raise ValueError("Unsafe package member")
    archive.extractall(destination)
del data
root = Path('/kaggle/working/video_engine')
log = root/'renderer.log'
tail = collections.deque(maxlen=70)
started = time.monotonic()
print("Prepared scene loaded. Checking GPU, then installing the verified renderer and character.", flush=True)
with log.open('w', encoding='utf-8') as logfile:
    proc = subprocess.Popen([sys.executable, '-u', str(root/'run_colab_render.py'), '--root', str(root)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        logfile.write(line)
        tail.append(line)
        if line.startswith(('GPU_STAGE', 'BUILD_STAGE', 'FRAME_PROGRESS', 'GPU_VIDEO_READY', 'GPU_ERROR')):
            print(line.rstrip(), flush=True)
    code = proc.wait()
if code:
    print(''.join(tail))
    report = root/'gpu-error-report.zip'
    with zipfile.ZipFile(report, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in [log, root/'GPU_ERROR.txt', root/'gpu-status.json']:
            if path.exists(): archive.write(path, path.name)
    print("Render failed. Download the linked error report and send it back to the chat.")
    download_link(report)
    raise RuntimeError(f"GPU render failed with exit code {code}. The complete log is in the error report.")
result = json.loads((root/'gpu-status.json').read_text())
output = Path(result['output'])
print(json.dumps({'status':'complete', 'total_seconds':round(time.monotonic()-started, 1),
                  'renderer':result['renderer'], 'render_and_encode_seconds':result['render_and_encode_seconds']}, indent=2))
display(Video(str(output), embed=True))
download_link(output)
print("Done. Send the video or the output above back to the chat.")
